In [1]:
import pandas as pd
from src.common import config

In [2]:
df = pd.read_csv(config.GLOBAL_SUPERSTORE_PATH, encoding="ISO-8859-1")

# Estructura y tipos
print(df.info())
print("\nPorcentaje de nulos por columna\n", df.isnull().mean())
print("\nNumero de duplicados en el DF\n", df.duplicated().sum())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 51290 entries, 0 to 51289
Data columns (total 24 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   Row ID          51290 non-null  int64  
 1   Order ID        51290 non-null  object 
 2   Order Date      51290 non-null  object 
 3   Ship Date       51290 non-null  object 
 4   Ship Mode       51290 non-null  object 
 5   Customer ID     51290 non-null  object 
 6   Customer Name   51290 non-null  object 
 7   Segment         51290 non-null  object 
 8   City            51290 non-null  object 
 9   State           51290 non-null  object 
 10  Country         51290 non-null  object 
 11  Postal Code     9994 non-null   float64
 12  Market          51290 non-null  object 
 13  Region          51290 non-null  object 
 14  Product ID      51290 non-null  object 
 15  Category        51290 non-null  object 
 16  Sub-Category    51290 non-null  object 
 17  Product Name    51290 non-null 

Postal Code presenta 80% de campos nulos, siendo candidata a descartarse. No es necesario a nivel pais.

In [24]:
# Rango de fechas
print("Rango Order Date: ", df['Order Date'].min(), " -> ", df['Order Date'].max())

# Formato de fechas
print("Order Date: ", df['Order Date'].max())
print("Ship Date: ", df['Ship Date'].max())


Rango Order Date:  01-01-2011  ->  31-12-2014
Order Date:  31-12-2014
Ship Date:  31-12-2014


Las fechas estan en formato dd-mm-YYYY mostrando consistencia en formatos

In [15]:
# Granularidad de fila
print("Valores duplicados en Row ID: ", df['Row ID'].duplicated().sum())
print("Valores duplicados en Order ID: ", df['Order ID'].duplicated().sum())

Valores duplicados en Row ID:  0
Valores duplicados en Order ID:  26255


In [18]:
# Jerarquia geografica
print(df.groupby(['Market', 'Region'])['Country'].unique())

Market  Region        
APAC    Central Asia      [Afghanistan, India, Bangladesh, Pakistan, Nep...
        North Asia        [China, Taiwan, Japan, South Korea, Hong Kong,...
        Oceania                  [Australia, New Zealand, Papua New Guinea]
        Southeast Asia    [Indonesia, Philippines, Malaysia, Thailand, C...
Africa  Africa            [Senegal, Tanzania, Democratic Republic of the...
Canada  Canada                                                     [Canada]
EMEA    EMEA              [Saudi Arabia, Poland, Iran, Ukraine, Belarus,...
EU      Central           [Germany, France, Austria, Netherlands, Belgiu...
        North             [United Kingdom, Finland, Sweden, Norway, Irel...
        South                                      [Italy, Spain, Portugal]
LATAM   Caribbean         [Dominican Republic, Cuba, Jamaica, Martinique...
        Central           [El Salvador, Nicaragua, Guatemala, Honduras, ...
        North                                                    

In [25]:
print(df['Country'].unique())

['United States' 'Australia' 'Germany' 'Senegal' 'New Zealand'
 'Afghanistan' 'Saudi Arabia' 'Brazil' 'China' 'France' 'Italy' 'Tanzania'
 'Poland' 'United Kingdom' 'Mexico' 'El Salvador' 'Taiwan' 'India'
 'Dominican Republic' 'Democratic Republic of the Congo' 'Indonesia'
 'Uruguay' 'Iran' 'Mozambique' 'Bangladesh' 'Spain' 'Ukraine' 'Nicaragua'
 'Morocco' 'Canada' 'Philippines' 'Austria' 'Colombia' 'Netherlands'
 'Malaysia' 'Ecuador' 'Thailand' 'Somalia' 'Guatemala' 'Belarus'
 'Cambodia' 'South Africa' 'Japan' 'Russia' 'Egypt' 'Azerbaijan'
 'Lithuania' 'Argentina' 'Lesotho' 'Vietnam' 'Cuba' 'Romania' 'Turkey'
 'Cameroon' 'Hungary' 'Singapore' 'Angola' 'Belgium' 'Pakistan' 'Finland'
 'Ghana' 'Zambia' 'Iraq' 'Liberia' 'Georgia' 'Switzerland' 'Albania'
 'Chad' 'Montenegro' 'Namibia' 'Portugal' 'Madagascar' 'Sweden'
 'Myanmar (Burma)' 'Jamaica' 'Qatar' 'Republic of the Congo' 'Norway'
 'Algeria' 'South Korea' 'Nigeria' 'Estonia' "Cote d'Ivoire" 'Honduras'
 'Paraguay' 'Czech Republic' 'Cen

In [21]:
print(df[df['Profit'] < 0]['Profit'])

1        -288.7650
3         -96.5400
9       -1862.3124
30       -148.4600
32       -350.4900
           ...    
51276      -4.4660
51278      -6.4560
51279     -21.1680
51280     -49.5720
51286      -1.1100
Name: Profit, Length: 12544, dtype: float64


Existen valores negativos en Profit. Es una venta real con perdida, no es un valor invalido.

In [9]:
df['Order Date'] = pd.to_datetime(df['Order Date'], format='%d-%m-%Y')
df['Ship Date'] = pd.to_datetime(df['Ship Date'], format='%d-%m-%Y')

check_dates = df[df['Ship Date'] < df['Order Date']]
print(check_dates)

Empty DataFrame
Columns: [Row ID, Order ID, Order Date, Ship Date, Ship Mode, Customer ID, Customer Name, Segment, City, State, Country, Postal Code, Market, Region, Product ID, Category, Sub-Category, Product Name, Sales, Quantity, Discount, Profit, Shipping Cost, Order Priority]
Index: []

[0 rows x 24 columns]


No hay fechas de Ship Date anteriores a Order Date (fechas ilogicas)

In [14]:
qty_negative = df[df['Quantity'] < 0]
min_discount = df['Discount'].min()
max_discount = df['Discount'].max()

print(qty_negative)
print(f"Discount Range: {min_discount} -> {max_discount}")

Empty DataFrame
Columns: [Row ID, Order ID, Order Date, Ship Date, Ship Mode, Customer ID, Customer Name, Segment, City, State, Country, Postal Code, Market, Region, Product ID, Category, Sub-Category, Product Name, Sales, Quantity, Discount, Profit, Shipping Cost, Order Priority]
Index: []

[0 rows x 24 columns]
Discount Range: 0.0 -> 0.85


No hay registos con Quantity negativo

El rango de discount es razonable

In [17]:
pd.set_option('display.max_columns', None)
profit_negative = df[df['Profit'] < 0]

print(profit_negative)

       Row ID         Order ID Order Date  Ship Date       Ship Mode  \
1       26341    IN-2013-77878 2013-02-05 2013-02-07    Second Class   
3       13524  ES-2013-1579342 2013-01-28 2013-01-30     First Class   
9       40936   CA-2012-116638 2012-01-28 2012-01-31    Second Class   
30        220   US-2011-128776 2011-12-28 2011-12-30    Second Class   
32      32735   CA-2012-139731 2012-10-15 2012-10-15        Same Day   
...       ...              ...        ...        ...             ...   
51276   31558   US-2014-155299 2014-06-09 2014-06-13  Standard Class   
51278   43818     NI-2011-4700 2011-09-08 2011-09-10    Second Class   
51279   46231     ZI-2011-4350 2011-03-21 2011-03-26  Standard Class   
51280   46582     TU-2014-6730 2014-11-29 2014-11-30     First Class   
51286   35398   US-2014-102288 2014-06-20 2014-06-24  Standard Class   

      Customer ID      Customer Name      Segment           City  \
1        JR-16210      Justin Ritter    Corporate     Wollongong   

In [19]:
print(df['Market'].unique())
print(df['Region'].unique())
print(df['Segment'].unique())
print(df['Ship Mode'].unique())
print(df['Order Priority'].unique())

['US' 'APAC' 'EU' 'Africa' 'EMEA' 'LATAM' 'Canada']
['East' 'Oceania' 'Central' 'Africa' 'West' 'South' 'Central Asia' 'EMEA'
 'North Asia' 'North' 'Caribbean' 'Southeast Asia' 'Canada']
['Consumer' 'Corporate' 'Home Office']
['Same Day' 'Second Class' 'First Class' 'Standard Class']
['Critical' 'Medium' 'High' 'Low']
